In [ ]:
#Este comando enlaza la notebook con Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import copy
import scipy
import random
import math
import matplotlib.pyplot as plt
import itertools

Encontrando rutas más cortas

In [ ]:
G = nx.DiGraph()

edges = [
    (0,1,3),(0,2,5),(0,3,8),
    (1,2,10),(1,4,9),
    (2,3,1),(2,5,1),
    (3,0,2),(3,6,3),
    (4,2,2),(4,7,1),
    (5,4,3),(5,8,5),
    (6,5,3),(6,9,3),
    (7,6,4),
    (8,7,4),
    (9,8,4)
]
G.add_weighted_edges_from(edges)

In [ ]:
for node, degree in G.out_degree(weight='weight'):
    print(f"Nodo {node}: {degree}")

In [ ]:
for node, degree in G.in_degree(weight='weight'):
    print(f"Nodo {node}: {degree}")

In [ ]:
#Dibujando en NetworkX
plt.figure()
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_size=700)
plt.title("Red dirigida con pesos en las aristas")
plt.show()

In [ ]:
#Obteniendo la matriz de adyacencia de flechas que salen
A = nx.adjacency_matrix(G).todense()
print('flechas que salen\n', A)
#Obteniendo la matriz de adyacencia de flechas que entran, solo transponer A
B = A.transpose()
print('flechas que entran\n', B)
# Verificando que la red sea fuertemente conexa
print('Fuertemente conexa =', nx.is_strongly_connected(G))

In [ ]:
#la red es fuertemente conexa
nx.is_strongly_connected(G)

In [ ]:
#Encontrando la distanca del nodo 1 al nodo 9
print("Distancia", nx.shortest_path_length(G, source=0, target=9))
#Encontrando la ruta más corta del nodo 1 al nodo 9
print("Ruta", nx.shortest_path(G, source=0, target=9))

In [ ]:
#Centralidad de eigenvector tomando en cuenta aristas pesadas
nx.eigenvector_centrality(G, weight='weight')

verificando que tipo de C.Eigenvector es la anterior mediante iteraciones de las matrices de adyacencia

In [ ]:
#Para verifiicar C. Eigenvector
def eigenvector_centrality(A,max_iter,importancias):
  pesos_iniciles = np.array(importancias)
  pesos_iniciles = np.transpose(pesos_iniciles)
  for iteracion in range(1,max_iter):
    pesos_iniciles = A@pesos_iniciles
  pesos_iniciles_normalized = pesos_iniciles / np.sqrt(np.sum(pesos_iniciles**2))
  return pesos_iniciles_normalized

In [ ]:
importancias = list(np.ones(A.shape[0]))
# = = la matriz de adyacencia de flechas que salen
print('flechas que salen\n',eigenvector_centrality(A, 18, importancias))
# = = la matriz de adyacencia de flechas que entran
print('\n flechas que entran\n', eigenvector_centrality(B, 18, importancias))

## Actividad

Obtener una lista de C.Exgrado pesado mediante algún comando de NetworkX

Luego lo mismo pero con C.Ingrado pesado

¿Cuáles son los 3 nodos más importantes según cada una de estas dos métricas?

# Haciendo dibujos bonitos

In [ ]:
#Instalando el paquete Pathpy (no incluido por default en Colab)
!pip install pathpy --quiet
import pathpy as pp

In [ ]:
'''Dibujando la red con Pathpy'''
aristas = list(G.edges())
Gra = pp.Network(directed=True)
for arista in  aristas:
  Gra.add_edge(arista[0],arista[1])
dict_size_nodos = {node: 3*G.degree(node) for node in G.nodes()}
style = {'width': 600, 'height': 600,
         'node_color': "tomato",
          'label_size': '16px','edge_width' : 1.5,
          'node_size': dict_size_nodos,
         'edge_color': "blue"}
pp.visualisation.plot(Gra,**style)

# Analizando una base de datos

Afluencia diaria de Metrobús CDMX (https://datos.cdmx.gob.mx/dataset/?q=Afluencia+metrobus+)

In [4]:
import pandas as pd
df=pd.read_excel('/content/drive/My Drive/Clases_Redes_Simples/para_clase_redes_bipartitas.xlsx')
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/Clases_Redes_Simples/para_clase_redes_bipartitas.xlsx'

In [ ]:
aristas = [(x['linea'],x['fecha'],x['afluencia']) for indice,x in df[1:210].iterrows()]
print(aristas)

In [ ]:
#Creando Gráfica vacía
B=nx.Graph()

In [ ]:
#Agregando nodos
B.add_nodes_from([arista[0] for arista in aristas], bipartite=0)
B.add_nodes_from([arista[1] for arista in aristas], bipartite=1)

In [ ]:
B.number_of_nodes()

In [ ]:
# Agregando Aristas
B.add_weighted_edges_from(aristas)

In [ ]:
B.number_of_edges()

In [ ]:
nx.degree(B)

In [ ]:
plt.figure(figsize=(10,10),dpi=70)
pos=nx.bipartite_layout(B, df['linea'],align='vertical')
nx.draw(B,pos,with_labels=True)
plt.show()

In [ ]:
for nodo in ['LÃ\xadnea 1','LÃ\xadnea 2', 'LÃ\xadnea 3', 'LÃ\xadnea 4']:
  print(B.degree(nodo, weight='weight'))


In [ ]:
nx.density(B)

In [ ]:
B.number_of_edges()*2/(B.number_of_nodes()*(B.number_of_nodes()-1))

## Actividad

1. Quitar el valor Nan o NaT de los datos y en consecuencia del dibujo de la red.

2. Obtener una lista de C.Grado pesado de cada línea mediante algún comando de NetworkX.

3. Dibujar la red con pathpy, poniendo diferente tamaño a los nodos de acuerdo a su grado, poner nodos en algún tono de verde y aritas en algún tono de morado.

4. ¿Cuáles son las 3 líneas que tuvieron mayor afluencia en junio?

5. ¿Cuáles son las 3 líneas que tuvieron mayor afluencia en julio?

6. ¿Cuáles son las 3 líneas que tuvieron mayor afluencia en ambos meses?

# Otro ejercicio

In [ ]:
def cargar_metro_cdmx():
    G = nx.Graph()

    # Definición de líneas (principales conexiones para análisis)
    lineas = {
        "L1": ["Pantitlán", "San Lázaro", "Candelaria", "Pino Suárez", "Salto del Agua", "Balderas", "Tacubaya", "Observatorio"],
        "L2": ["Tasqueña", "Chabacano", "Pino Suárez", "Bellas Artes", "Hidalgo", "Tacuba", "Cuatro Caminos"],
        "L3": ["Universidad", "Zapata", "Centro Médico", "Balderas", "Hidalgo", "La Raza", "Deportivo 18 de Marzo", "Indios Verdes"],
        "L7": ["Barranca del Muerto", "Mixcoac", "Tacubaya", "Tacuba", "El Rosario"],
        "L9": ["Pantitlán", "Chabacano", "Centro Médico", "Tacubaya"],
        "L12": ["Tláhuac", "Atlalilco", "Ermita", "Zapata", "Mixcoac"],
        "LB": ["Buenavista", "Guerrero", "Garibaldi", "Oceanía", "Ciudad Azteca"]
    }

    # Agregar conexiones al grafo
    for nombre_linea, estaciones in lineas.items():
        for i in range(len(estaciones) - 1):
            G.add_edge(estaciones[i], estaciones[i+1], linea=nombre_linea)

    return G

# Crear y visualizar rápidamente
G_metro = cargar_metro_cdmx()
print(f"Red cargada: {G_metro.number_of_nodes()} estaciones y {G_metro.number_of_edges()} conexiones.")

# Visualización básica
plt.figure(figsize=(10, 7))
nx.draw(G_metro, with_labels=True, node_color='gold', node_size=800, font_size=8)

plt.title("Esqueleto de Conexiones Principales - Metro CDMX")
plt.show()

In [ ]:
#Obteniendo orden y tamaño
print('orden  ',nx.number_of_nodes(G_metro))
print('tamaño  ', nx.number_of_edges(G_metro))
#Viendo si la red es conexa
print('red conexa  = ',nx.is_connected(G_metro))
#Numero de componentes conexas
print('numero de componentes conexas ',nx.number_connected_components(G_metro))
#Tamaño de las componentes
componentes = nx.connected_components(G_metro)
longitudes_componentes = []
for componente in componentes:
  longitudes_componentes.append(len(componente))
print('longitudes de las componentes', longitudes_componentes)

## Actividad une la red para que sea conexa de tal manera que el diamétro de la red resultante sea lo más chico posible. Posteriormente calcular C: intermedia, de cercanía, Eigenvector.

In [ ]:
#Creando una red copia de la anterior
G_metro_2 = copy.deepcopy(G_metro)
#Agregando conexiones
G_metro_2.add_edge("Buenavista", "Tasqueña")
print('red conexa  = ',nx.is_connected(G_metro_2))
#Imprimiendo el diamétro
print('diametro = ', nx.diameter(G_metro_2))
# Visualización básica
plt.figure(figsize=(7, 7))
nx.draw(G_metro_2, with_labels=True, node_color='gold', node_size=700, font_size=8)
plt.title("Esqueleto de Conexiones Principales - Metro CDMX")
plt.show()

In [ ]:
#Creando una red copia de la anterior
G_metro_2 = copy.deepcopy(G_metro)
#Agregando conexiones
G_metro_2.add_edge("Buenavista", "Atlalilco")
print('red conexa  = ',nx.is_connected(G_metro_2))
#Imprimiendo el diamétro
print('diametro = ', nx.diameter(G_metro_2))
# Visualización básica
plt.figure(figsize=(7, 7))
nx.draw(G_metro_2, with_labels=True, node_color='#90ee90', node_size=700, font_size=8)
plt.title("Esqueleto de Conexiones Principales - Metro CDMX")
plt.show()

# Actividad del metro 2

1) Calcular la ruta más corta antes y despues de quitar Pino suarez entre las estaciones "Tasqueña" y "Bellas Artes"

In [ ]:
def cargar_metro_dirigido():
    # Cambiamos Graph por DiGraph
    G = nx.DiGraph()
    lineas = {
        "L1": ["Pantitlán", "San Lázaro", "Candelaria", "Pino Suárez", "Salto del Agua", "Balderas", "Tacubaya", "Observatorio"],
        "L2": ["Tasqueña", "Chabacano", "Pino Suárez", "Bellas Artes", "Hidalgo", "Tacuba", "Cuatro Caminos"],
        "L3": ["Universidad", "Zapata", "Centro Médico", "Balderas", "Hidalgo", "La Raza", "Deportivo 18 de Marzo", "Indios Verdes"],
        "L7": ["Barranca del Muerto", "Mixcoac", "Tacubaya", "Tacuba", "El Rosario"],
        "L9": ["Pantitlán", "Chabacano", "Centro Médico", "Tacubaya"],
        "L12": ["Tláhuac", "Atlalilco", "Ermita", "Zapata", "Mixcoac"],
        "LB": ["Buenavista", "Guerrero", "Garibaldi", "Oceanía", "Ciudad Azteca"]
    }

    for nombre_linea, estaciones in lineas.items():
        for i in range(len(estaciones) - 1):
            u, v = estaciones[i], estaciones[i+1]

            # Agregamos la dirección "hacia adelante"
            G.add_edge(u, v, linea=nombre_linea)

            # Simulamos que solo el 70% de las vías están abiertas en ambos sentidos
            # Esto crea "trampas" o "sumideros"
            # Queremos una semilla de aleatoriedad para que la red sea la misma cada que se corra el código
            random.seed(i+5)
            if random.random() < 0.65:
                G.add_edge(v, u, linea=nombre_linea)

    return G

G_dir = cargar_metro_dirigido()

# Cálculo de métricas para la clase
pagerank = nx.pagerank(G_dir)
katz = nx.katz_centrality(G_dir, alpha=0.1, beta=1.0)

print(f"Red Dirigida: {G_dir.number_of_nodes()} nodos y {G_dir.number_of_edges()} aristas dirigidas.")
#Dibujo de la red
plt.figure(figsize=(11, 11))
pos = nx.spring_layout(G_dir, seed=42)
nx.draw(G_dir, pos, with_labels=True, node_size=500, font_size=8, node_color="gold", edge_color="gray", arrows=True, width=0.9)

## Actividad
0. Encuentra la distancia y la geodésica de Tláhuac a Pino suarez.
1. Calcula PageRank y Katz y determina los 3 nodos más importantes en orden descendente para estas métricas.
2. Comparen el nodo con mayor Katz  frente al de mayor Grado de Entrada (in-degree) ¿coinciden? ¿Por qué crees que sucede esto?


# Modelos nulos

# Modelos nulos
Grafo del Club de Karate de Zachary. Es una red social real donde los nodos son miembros del club y las aristas son amistades. Como es una red social humana, está llena de triángulos


La red real tiene 45 triángulos, pero el modelo nulo (dependiendo de la corrida, ya que es aleatorio) suele tener solo unos 4 o 5 triángulos.

La validación estadística: Esto te demuestra que los 45 triángulos originales no son producto de tener nodos muy conectados (hubs). Si fuera solo por eso, el modelo nulo también tendría 45. Esos triángulos existen porque hay una "fuerza" subyacente (afinidad, lógica semántica, o proximidad espacial en el caso de las redes de transporte urbano) que hace que se agrupen.

In [ ]:
# 1. Cargar la red real (El Club de Karate)
G_real = nx.karate_club_graph()
print(f"Red Real: {G_real.number_of_nodes()} nodos")
# 2. Extraer la "secuencia de grados"
grados = [grado for nodo, grado in G_real.degree()]
print(grados)
# 3. Crear el Modelo Nulo (Configuration Model)
G_nulo_multi = nx.configuration_model(grados)
G_nulo = nx.Graph(G_nulo_multi) # Convertir a grafo simple
G_nulo.remove_edges_from(nx.selfloop_edges(G_nulo)) # Quitar auto-conexiones
print(f"Red Nula: {G_nulo.number_of_nodes()} nodos")
grados_nulo = [grado for nodo, grado in G_nulo.degree()]
print(grados_nulo)
# --- NUEVA SECCIÓN: VISUALIZACIÓN ---

# Crear una figura ancha para poner los dos gráficos lado a lado
plt.figure(figsize=(16, 7))

# Layout: spring_layout simula resortes en las aristas, lo que junta a los
# nodos muy conectados y empuja a los desconectados.
pos_real = nx.spring_layout(G_real, seed=42)
pos_nulo = nx.spring_layout(G_nulo, seed=42)

# Subgráfico 1: Red Real
plt.subplot(1, 2, 1)
plt.title("Red Real (Club de Karate)\nEstructura clara y triángulos visibles", fontsize=14, fontweight='bold')
nx.draw(G_real, pos=pos_real, with_labels=True,
        node_color='#add8e6', edge_color='#a9a9a9',
        node_size=500, font_size=9, font_weight='bold')

# Subgráfico 2: Modelo Nulo
plt.subplot(1, 2, 2)
plt.title("Modelo Nulo (Configuration Model)\nEstructura destruida (aspecto de 'bola de pelo')", fontsize=14, fontweight='bold')
nx.draw(G_nulo, pos=pos_nulo, with_labels=True,
        node_color='#90ee90', edge_color='#a9a9a9',
        node_size=500, font_size=9, font_weight='bold')

# Ajustar márgenes y mostrar
plt.tight_layout()
plt.show()

En un momento determinado, los miembros de esta red social se polarizaron en dos facciones diferentes debido a una discusión entre el instructor y el presidente. Estas dos facciones, representadas en la Figura con diferentes colores, actúan como grupos cohesivos que pueden considerarse entidades independientes en la red.

En la red de Delfines, se establecen vínculos entre dos animales si se les ve juntos con más frecuencia de la esperada al azar. Estos delfines se dividieron en grupos después de que un delfín en particular se alejara por un tiempo.

La polarización política en EE. UU. genera diversas redes agrupadas. Ilustramos esto con un ejemplo basado en la literatura política. La Figura 21.2 representa una red de libros sobre política estadounidense publicados en torno a las elecciones presidenciales de 2004 y vendidos en Amazon.com. Existe un vínculo entre dos libros si comparten varios compradores comunes. Como se puede observar, existe una clara convergencia en dos comunidades principales. Estas representan las compras de los consumidores de literatura conservadora, por un lado, y de literatura liberal, por el otro.

Es decir, las comunidades están conectadas internamente. Podemos definir, de forma general, un grupo o comunidad de la siguiente manera.

**A cluster (or community) of a network is an internally connected set of nodes for which the internal density is significantly larger than the external one.**

In [ ]:
import networkx as nx
import itertools

# Crear grafo de ejemplo
G = nx.Graph()

edges = [
    (1,2),(2,3),(1,4),(2,5),(3,6),
    (4,5),(5,6)
]

G.add_edges_from(edges)

# Partición inicial
A = {1,2,3}
B = {4,5,6}

def compute_D(G,A,B):
    D={}
    for v in G.nodes():
        E=0
        I=0
        for u in G.neighbors(v):
            if (v in A and u in B) or (v in B and u in A):
                E+=1
            else:
                I+=1
        D[v]=E-I
    return D

def gain(G,a,b,D):
    cab = 1 if G.has_edge(a,b) else 0
    return D[a]+D[b]-2*cab


def kernighan_lin_step(G,A,B):

    A=set(A)
    B=set(B)

    locked=set()

    gains=[]
    pairs=[]

    D=compute_D(G,A,B)

    print("\nD values:",D)

    for i in range(len(A)):

        best=None
        best_gain=-999

        for a in A-locked:
            for b in B-locked:

                g=gain(G,a,b,D)

                if g>best_gain:
                    best_gain=g
                    best=(a,b)

        a,b=best

        print(f"\nIteración {i+1}")
        print("Swap candidato:",a,b)
        print("Ganancia:",best_gain)

        gains.append(best_gain)
        pairs.append((a,b))

        locked.add(a)
        locked.add(b)

        # actualizar D
        for v in G.nodes():
            if v not in locked:
                if v in A:
                    D[v]+=2*(G.has_edge(v,a)-G.has_edge(v,b))
                else:
                    D[v]+=2*(G.has_edge(v,b)-G.has_edge(v,a))

    print("\nGains acumuladas:",gains)

    # encontrar mejor prefijo
    cum=[sum(gains[:i+1]) for i in range(len(gains))]
    k=max(range(len(cum)), key=lambda i:cum[i])

    print("Mejor k:",k+1)

    for i in range(k+1):
        a,b=pairs[i]
        A.remove(a)
        B.remove(b)
        A.add(b)
        B.add(a)

    return A,B


print("Partición inicial")
print("A:",A)
print("B:",B)

A,B=kernighan_lin_step(G,A,B)

print("\nPartición final")
print("A:",A)
print("B:",B)

# Otro ejemplo

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# -------------------------
# Crear grafo de ejemplo
# -------------------------

G = nx.Graph()

edges = [
    (1,2),(2,3),(2,4),(2,8),(3,5),(3,6),
    (4,6),(6,8),(5,7),(5,9), (5,10),  (8,10), (10,9), (10,11), (10,12), (12,11),(11,9),
(9,7)]

G.add_edges_from(edges)
# Partición inicial
A = {1,2,3, 4,5,6}
B = {7,8,9,10,11,12}
# layout fijo
pos = nx.spring_layout(G, seed=1)


# -------------------------
# Dibujar partición
# -------------------------

def draw_partition(G, A, B, pos, title=""):

    colors=[]

    for node in G.nodes():
        if node in A:
            colors.append("skyblue")
        else:
            colors.append("salmon")

    plt.figure()
    nx.draw(
        G,
        pos,
        with_labels=True,
        node_color=colors,
        node_size=800,
        edge_color="gray"
    )

    plt.title(title)
    plt.show()

# -------------------------
# calcular D
# -------------------------

def compute_D(G,A,B):

    D={}

    for v in G.nodes():

        E=0
        I=0

        for u in G.neighbors(v):

            if (v in A and u in B) or (v in B and u in A):
                E+=1
            else:
                I+=1

        D[v]=E-I

    return D


# -------------------------
# ganancia
# -------------------------

def gain(G,a,b,D):

    cab = 1 if G.has_edge(a,b) else 0

    return D[a]+D[b]-2*cab


# -------------------------
# algoritmo
# -------------------------

def kernighan_lin_step(G,A,B,pos):

    A=set(A)
    B=set(B)

    locked=set()

    gains=[]
    pairs=[]

    D=compute_D(G,A,B)

    print("\nD values:",D)

    draw_partition(G,A,B,pos,"Partición inicial")

    for i in range(len(A)):

        best=None
        best_gain=-999

        for a in A-locked:
            for b in B-locked:

                g=gain(G,a,b,D)

                if g>best_gain:
                    best_gain=g
                    best=(a,b)

        a,b=best

        print(f"\nIteración {i+1}")
        print("Swap candidato:",a,b)
        print("Ganancia:",best_gain)

        gains.append(best_gain)
        pairs.append((a,b))

        locked.add(a)
        locked.add(b)

        # mostrar swap candidato
        A_temp = A.copy()
        B_temp = B.copy()

        A_temp.remove(a)
        B_temp.remove(b)

        A_temp.add(b)
        B_temp.add(a)

        draw_partition(
            G,
            A_temp,
            B_temp,
            pos,
            f"Iteración {i+1}: swap ({a},{b})"
        )

        # actualizar D
        for v in G.nodes():

            if v not in locked:

                if v in A:
                    D[v]+=2*(G.has_edge(v,a)-G.has_edge(v,b))
                else:
                    D[v]+=2*(G.has_edge(v,b)-G.has_edge(v,a))

    print("\nGains:",gains)

    cum=[sum(gains[:i+1]) for i in range(len(gains))]

    k=max(range(len(cum)), key=lambda i:cum[i])

    print("Mejor prefijo:",k+1)

    for i in range(k+1):

        a,b=pairs[i]

        A.remove(a)
        B.remove(b)

        A.add(b)
        B.add(a)

    draw_partition(G,A,B,pos,"Partición final")

    return A,B


# -------------------------
# ejecutar
# -------------------------

print("Partición inicial")
print(A,B)

A,B = kernighan_lin_step(G,A,B,pos)

print("\nPartición final")
print(A,B)

In [ ]:
# Crear grafo de ejemplo
G = nx.Graph()

edges = [
    (1,2),(2,3),(2,4),(2,8),(3,5),(3,6),
    (4,6),(6,8),(5,7),(5,9), (5,10),  (8,10), (10,9), (10,11), (10,12), (12,11),(11,9),
(9,7)]

G.add_edges_from(edges)

# Partición inicial
A = {1,2,3, 4,5,6}
B = {7,8,9,10,11,12 }

def compute_D(G,A,B):
    D={}
    for v in G.nodes():
        E=0
        I=0
        for u in G.neighbors(v):
            if (v in A and u in B) or (v in B and u in A):
                E+=1
            else:
                I+=1
        D[v]=E-I
    return D

def gain(G,a,b,D):
    cab = 1 if G.has_edge(a,b) else 0
    return D[a]+D[b]-2*cab


def kernighan_lin_step(G,A,B):

    A=set(A)
    B=set(B)

    locked=set()

    gains=[]
    pairs=[]

    D=compute_D(G,A,B)

    print("\nD values:",D)

    for i in range(len(A)):

        best=None
        best_gain=-999

        for a in A-locked:
            for b in B-locked:

                g=gain(G,a,b,D)

                if g>best_gain:
                    best_gain=g
                    best=(a,b)

        a,b=best

        print(f"\nIteración {i+1}")
        print("Swap candidato:",a,b)
        print("Ganancia:",best_gain)

        gains.append(best_gain)
        pairs.append((a,b))

        locked.add(a)
        locked.add(b)

        # actualizar D
        for v in G.nodes():
            if v not in locked:
                if v in A:
                    D[v]+=2*(G.has_edge(v,a)-G.has_edge(v,b))
                else:
                    D[v]+=2*(G.has_edge(v,b)-G.has_edge(v,a))

    print("\nGains acumuladas:",gains)

    # encontrar mejor prefijo
    cum=[sum(gains[:i+1]) for i in range(len(gains))]
    k=max(range(len(cum)), key=lambda i:cum[i])

    print("Mejor k:",k+1)

    for i in range(k+1):
        a,b=pairs[i]
        A.remove(a)
        B.remove(b)
        A.add(b)
        B.add(a)

    return A,B


print("Partición inicial")
print("A:",A)
print("B:",B)

A,B=kernighan_lin_step(G,A,B)

print("\nPartición final")
print("A:",A)
print("B:",B)

# Ejemplo Red Zachary Karate Club

In [ ]:
from networkx.algorithms.community import girvan_newman
from networkx.algorithms.community.quality import modularity
G = nx.karate_club_graph()

# Ejecutar Girvan-Newman
comp = girvan_newman(G)

partitions = []
mod_values = []

# Guardar primeras 6 iteraciones
for communities in itertools.islice(comp, 6):
    communities = [set(c) for c in communities]
    partitions.append(communities)
    mod_values.append(modularity(G, communities))

# Mostrar modularidad
for i, q in enumerate(mod_values):
    print(f"Iteración {i+1}: modularidad = {q:.4f}")

# Tomar partición de la tercera iteración
communities = partitions[2]

# Layout para graficar
pos = nx.spring_layout(G, seed=42)

# Colorear nodos según comunidad
colors = []
for node in G.nodes():
    for i, comm in enumerate(communities):
        if node in comm:
            colors.append(i)
            break

# Dibujar red
plt.figure(figsize=(6,6))
nx.draw(G, pos,
        node_color=colors,
        with_labels=True,
        cmap=plt.cm.Set3,
        node_size=400)

plt.title("Karate Club Network\nCommunities after 3 Girvan–Newman iterations")
plt.show()

# Graficar evolución de modularidad
plt.figure(figsize=(6,4))
plt.plot(range(1,len(mod_values)+1), mod_values, marker='o')
plt.xlabel("Iteration")
plt.ylabel("Modularity Q")
plt.title("Evolution of modularity during Girvan–Newman")
plt.grid(True)
plt.show()

# Ejemplo Red artificial del metro

In [ ]:
#Creando una red copia de la anterior
G_metro_2 = copy.deepcopy(G_metro)
#Agregando conexiones
G_metro_2.add_edge("Buenavista", "Tasqueña")
print('red conexa  = ',nx.is_connected(G_metro_2))
#Imprimiendo el diamétro
print('diametro = ', nx.diameter(G_metro_2))
# Visualización básica
plt.figure(figsize=(10, 6))
pos = nx.spring_layout(G_metro_2, seed=42)
nx.draw(G_metro_2,pos, with_labels=True, node_color='gold', node_size=700, font_size=8)
plt.title("Esqueleto de Conexiones - Metro CDMX")
plt.show()

Algoritmo de Girvan–Newman en Python

**Ventajas**

1. Conceptualmente muy claro.

2. Detecta estructura jerárquica de comunidades.

3. Funciona bien en redes pequeñas o medianas.

**Desventajas**

1. Computacionalmente costoso, $O(N^3)$.

2. No escala bien a redes muy grandes.

**Cuando la red ya está separada en componentes:**

El algoritmo deja de buscar puentes entre comunidades grandes (porque ya no existen).

Empieza a detectar subcomunidades dentro de cada componente.

*Por eso el resultado final del algoritmo no es solo una partición, sino una jerarquía completa de divisiones.*

✅ Girvan–Newman detecta comunidades dentro de comunidades, lo cual es útil si en una red social existen subgrupos ideológicos dentro de grupos más grandes.

In [ ]:
# 1. Ejecutar Girvan-Newman
# Este algoritmo devuelve un iterador de particiones
comp_iter = community.girvan_newman(G_metro_2) #Comando para preparar la red
#Cada vez que escribimos next(comp_iter) hacemos una iteración
for iteracion in range(3):
  next(comp_iter)
first_partition = tuple(sorted(c) for c in next(comp_iter))
print(first_partition)
# 2. Calcular Modularidad
mod = community.modularity(G_metro_2, first_partition)
# 3. Dibujar
def dibujar_comunidades(G, partition, titulo):
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(10, 6))

    # Crear un diccionario de colores para cada comunidad
    colors = ['#FF5733', '#33FF57', '#3357FF', '#F333FF', '#FFFD33']
    node_color_map = {}

    for idx, cluster in enumerate(partition):
        color = colors[idx % len(colors)]
        for node in cluster:
            node_color_map[node] = color

    node_colors = [node_color_map[n] for n in G.nodes()]

    nx.draw(G, pos, node_color=node_colors, with_labels=True,
            node_size=600, font_size=8, edge_color='gray', arrows=True)
    plt.title(f"{titulo} | Modularidad: {mod:.4f}")
    plt.show()

dibujar_comunidades(G_metro_2, first_partition, "Comunidades: Girvan-Newman")

Para decidir cuántas comunidades existen, se suele usar la modularidad. De esta manera, la modularidad es una medida de la calidad del los clusters


